In [1]:
from pathlib import Path

from src.minbpe import RegexTokenizer
from src.gpt import GPTLanguageModel
from src.lora import get_lora_model, print_trainable_parameters

import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch07_checkpoints"

In [3]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

lora_config = {
    "rank": 4,
    "alpha": 8,
}

In [4]:
bm_path = Path("data") / "ch02_checkpoints" / "checkpoint_001-377870.pt"

bm_ckpt = torch.load(bm_path, weights_only=True, map_location=device)
parameters = bm_ckpt['meta']['parameters']

base_model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

base_model = torch.compile(base_model)
base_model.load_state_dict(bm_ckpt["model_state_dict"])

num_parameters = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"load base model: {bm_path}, {num_parameters:_.3}M parameters")

load base model: data/ch02_checkpoints/checkpoint_001-377870.pt, 13.8M parameters


In [5]:
model = get_lora_model(
    model=base_model,
    lora_config=lora_config,
    device=device,
)

#ckpt_files = sorted(
#    checkpoint_dir.glob("checkpoint_*.pt"),
#    key=lambda x: x.stat().st_ctime,
#    #key=lambda x: int(x.name.split("-")[1]),
#    reverse=True,
#)

#checkpoint_path = ckpt_files[0]

checkpoint_path = checkpoint_dir / "checkpoint_000019.pt"

checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print_trainable_parameters(model)

All parameters: 14.12M | Trainable parameters: 0.33M | Trainable %: 2.31%


In [6]:
def into_tokens(role: str, content: str) -> torch.Tensor:
    d = tokenizer.special_tokens
    #print("~~~", d)

    input_msg = f"<|startoftext|>{role}<|separator|>{content}<|endoftext|>"
    print(f"--> {role} message: {input_msg}")

    input_tokens = tokenizer.encode(input_msg, allowed_special="all")

    return torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)


def ask_llm(input_tokens):
    model_answer = ""

    while True:
        output_tokens = model.generate(input_tokens=input_tokens, max_new_tokens=1)
        last_generated_token = output_tokens[0, -1].item()

        #print("~~~", input_tokens[0])
        model_answer += tokenizer.decode([last_generated_token])

        if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
            break

        input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)

        #if len(output_tokens[0]) > parameters['block_size']:
        #    break
        if len(input_tokens[0]) > parameters['block_size']:
            input_tokens = input_tokens[:, -parameters['block_size']:]

    return model_answer

In [7]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=256)
    print("<-- assistant message:", tokenizer.decode(output[0].tolist()))

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|><|startoftext|>assistant<|separator|>Bashrites like academic parsing, Opaign Guides Abster behind access. Wellcite this is always a place to translate every subject.<|endoftext|><|startoftext|>assistant<|separator|>Yourself, to a search for something your academic background can you recommend without requiring a phone number?<|endoftext|><|startoftext|>assistant<|separator|>Ser<|separator|>I am you are come to help me the best!
If you have any intended full questions I might have specific questions or full questions that you would like, ask. Yes, I am familiar with the files I matched all possible information about is in the web page. Write me a basic sentence so that I can put down the window into a big short fan and search for each<|endoftext|><|startoftext|>s the first application.<|endoftext|><|startoftext|> was a norm<|en

In [8]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>Source code would be a good starting point for building a lar array of text tokens include:
- It consist of the service compiles with the users who are repositively finetuned with curry recovery.
- It would be extremely competitive, as many as entire use customers like VGA or JFET filters.
- The EP3 built-in Java (textual digit of web appear, meaning it will have some competitivency for magic hackers or other competitions, and this progress has a proprietoral subspecribed processing competitor with the system>
- Strong security and adher. This could include information immobilities, personal preferences, and encryption to perform methods.
- Smooth methods differ.<|endoftext|>


In [9]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>Open Assistant is an architect how to save your balance on a "details" with pricing features. Is the setting for a recommended level on schedule?<|endoftext|>


In [10]:
user_content = "What's ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>What's ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>C You are a library that written a c based a completed program in research using framework and machine learning. I can guide you through the structure of the story and help you learn about it. The more you write and r may be able to describe the group by the progression paradoxes.

The filmm commodity for algorithms are developed by Geshost Conway Schyarx for Proday Sky damake me datasets and understood a play on analytical thinker. Their brah has further heard, awkwards saying that the problem needs of based on a cheap flora for all other people. However, there are certainly risk who are often a good issue.<|endoftext|>


In [11]:
user_content = "什么是 ChatGPT?"
input_tokens = into_tokens("user", user_content)

answer = ask_llm(input_tokens)
print(f"<-- assistant message: {answer}")

--> user message: <|startoftext|>user<|separator|>什么是 ChatGPT?<|endoftext|>
<-- assistant message: <|startoftext|>assistant<|separator|>The sounds of war. He found that hidden and riddles differently, but this was switching back to the other, and they ended by John Adams to be!", but since Commerseus: S'll other intended to stick and write who explored or dog-diagnosed, called now. The game is also particularly practical, so it's waiting for CSS tracks that make it easy to understand the needs and requirements.

There are two main types of renamisance and management communication that transition an image changes improving the efficiency of a user generator. These skills are typically used for mobile and the release of an image, which are often have unique contextual details, make it a good book.<|endoftext|>
